# Synthetic Kannada caller turns (#7)

Runs `training/generate_data.py`: labels are sampled from `data/generation/scenarios.json`, an instruction-tuned model writes each caller sentence in spoken Kannada, and only sentences that pass the consistency checks are kept.

**Before running**
- Kaggle: Settings → Accelerator **GPU T4 x2**, Internet **on**; Add-ons → Secrets → `HF_TOKEN` (a write token).
- Colab: Runtime → **T4 GPU**; Secrets (key icon) → `HF_TOKEN`, notebook access on. One T4 loads the model in 4-bit.
- The output goes to the private dataset repo `Dhwanidesk/dhwani-desk-data` (`DATASET_REPO` below).

Step 1 is a 40-row pilot per candidate model, so a Kannada reader can compare them before the long run.

In [ ]:
BRANCH = 'main'  # or a feature branch, to try changes before they merge
!git clone --depth 1 -b {BRANCH} https://github.com/Pranav-0710/dhwani-desk.git
%cd dhwani-desk/training
# transformers 4.56+ reads the `dtype` argument; older versions ignore it and load in fp32.
!pip install -q 'transformers>=4.56' accelerate bitsandbytes huggingface_hub

In [ ]:
import os

import torch

try:
    from kaggle_secrets import UserSecretsClient

    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except ImportError:
    from google.colab import userdata

    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

DATASET_REPO = 'Dhwanidesk/dhwani-desk-data'  # private; see STRUCTURE.md
ONE_GPU = torch.cuda.device_count() < 2
print(torch.cuda.device_count(), 'GPU(s)')

## Step 1: pilot, 40 rows per candidate

First pilot (2026-10-13): Qwen2.5-7B wrote mostly non-Kannada word strings; Qwen3-8B was readable but about a third of its sentences said something other than what was asked (a light "on" instead of "off", a wrong word for monkeys). This pilot tries bigger models: Qwen3-14B and Sarvam-M (24B, trained for Indian languages including Kannada). Both load in 4-bit, about 9 GB and 14 GB.

Each sample prints the sentence next to the scenario it was asked to describe. Read them against each other: the English gloss is not a check, because the model writes it from the request rather than translating its own sentence. The model whose sentences most often say what was asked, in natural spoken Kannada, wins, even at a lower pass rate.

In [ ]:
# name -> (model, needs 4-bit even on T4 x2); anything over ~9B params does not fit 2 x 15 GB in fp16
CANDIDATES = {'qwen3-14b': ('Qwen/Qwen3-14B', True), 'sarvam-m': ('sarvamai/sarvam-m', True)}
for short, (model, big) in CANDIDATES.items():
    four_bit = '--four-bit' if big or ONE_GPU else ''
    !python generate_data.py --n 40 --model {model} --out out/pilot-{short} {four_bit}
    # The downloads are ~30 GB and ~47 GB; Kaggle's disk does not hold both at once.
    !rm -rf ~/.cache/huggingface/hub/models--*

In [ ]:
import json

for short in CANDIDATES:
    with open(f'out/pilot-{short}/report.json', encoding='utf-8') as f:
        report = json.load(f)
    print()
    print(f"== {short}: kept {report['kept']}/{report['requested']}, "
          f"first-try pass {report['first_attempt_pass_rate']:.0%}, {report['rejection_reasons']}")
    with open(f'out/pilot-{short}/train.jsonl', encoding='utf-8') as f:
        rows = [json.loads(line) for line in f]
    for row in rows[:20]:
        print(f"{row['id']} | asked: {row['meta']['scenario']}")
        print(f"          wrote: {row['text']}")

## Step 2: full run and upload

Set `MODEL` to the pilot winner. 2,500 rows take a few hours on T4 x2 in 4-bit (Kaggle allows 12 hours per session). The run writes `train.jsonl`, `rejected.jsonl`, `report.json` and a dataset card, then uploads them to `DATASET_REPO`. Tagging the version and the 10% hand check follow in #12.

In [ ]:
MODEL, BIG = 'sarvamai/sarvam-m', True  # the pilot winner
four_bit = '--four-bit' if BIG or ONE_GPU else ''
!python generate_data.py --n 2500 --model {MODEL} --out out/train {four_bit} --push-repo {DATASET_REPO}